In [1]:
## Assignment 1 - MTE 544, Sachin Gill
## Completing the matrix multiplication for part A of question 1
print ("Part A: Matrix Multiplication")

Part A: Matrix Multiplication


In [2]:
import sympy as sp

# Symbolic variables (angles in radians)
theta, alpha, gamma = sp.symbols("theta alpha gamma", real=True)
x, y = sp.symbols("x y", real=True)
l_B, l_1, l_2 = sp.symbols("l_B l_1 l_2", positive=True)

def G(angle, tx, ty):
    """Planar homogeneous transformation."""
    return sp.Matrix([
        [sp.cos(angle), -sp.sin(angle), tx],
        [sp.sin(angle),  sp.cos(angle), ty],
        [0,              0,             1]
    ])

def simplify_matrix(M):
    return M.applyfunc(lambda entry: sp.trigsimp(sp.simplify(entry)))

# Naming: G_E_2 means G_E^2, converting E coordinates into frame 2.
G_E_2 = G(-sp.pi / 2, l_2, 0)
G_2_1 = G(-gamma, l_1, 0)
G_1_B = G(alpha, l_B, 0)
G_B_s = G(theta, x, y)

# Compose transformations: rightmost acts first.
G_E_s = simplify_matrix(G_B_s * G_1_B * G_2_1 * G_E_2)

for name, matrix in [
    ("G_E^2", G_E_2),
    ("G_2^1", G_2_1),
    ("G_1^B", G_1_B),
    ("G_B^s", G_B_s),
    ("G_E^s", G_E_s),
]:
    print(f"\n{name} =")
    sp.pprint(matrix)

print("\nFinal simplified matrix G_E^s:")
sp.pprint(G_E_s, use_unicode=True, wrap_line=False)


G_E^2 =
⎡0   1  l₂⎤
⎢         ⎥
⎢-1  0  0 ⎥
⎢         ⎥
⎣0   0  1 ⎦

G_2^1 =
⎡cos(γ)   sin(γ)  l₁⎤
⎢                   ⎥
⎢-sin(γ)  cos(γ)  0 ⎥
⎢                   ⎥
⎣   0       0     1 ⎦

G_1^B =
⎡cos(α)  -sin(α)  l_B⎤
⎢                    ⎥
⎢sin(α)  cos(α)    0 ⎥
⎢                    ⎥
⎣  0        0      1 ⎦

G_B^s =
⎡cos(θ)  -sin(θ)  x⎤
⎢                  ⎥
⎢sin(θ)  cos(θ)   y⎥
⎢                  ⎥
⎣  0        0     1⎦

G_E^s =
⎡sin(α - γ + θ)   cos(α - γ + θ)  l₁⋅cos(α + θ) + l₂⋅cos(α - γ + θ) + l_B⋅cos( ↪
⎢                                                                              ↪
⎢-cos(α - γ + θ)  sin(α - γ + θ)  l₁⋅sin(α + θ) + l₂⋅sin(α - γ + θ) + l_B⋅sin( ↪
⎢                                                                              ↪
⎣       0               0                                 1                    ↪

↪ θ) + x⎤
↪       ⎥
↪ θ) + y⎥
↪       ⎥
↪       ⎦

Final simplified matrix G_E^s:
⎡sin(α - γ + θ)   cos(α - γ + θ)  l₁⋅cos(α + θ) + l₂⋅cos(α - γ + θ) + l_B⋅cos

In [3]:
## Part B
## Then plug in the provided values 

configuration = {
    theta: -sp.pi / 2,
    alpha:  sp.pi / 6,
    gamma:  sp.pi / 4,
    l_1: 1,
    l_2: 1,
    l_B: sp.Rational(1, 2),
    x: sp.Rational(4, 5),
    y: sp.Rational(4, 5)
}

G_E_s_config = simplify_matrix(G_E_s.subs(configuration))

print("G_E^s at the given configuration:")
sp.pprint(G_E_s_config.evalf(6), wrap_line=False)

# Points expressed in frame E, with a homogeneous coordinate of 1
p1_E = sp.Matrix([0, 0, 1])
p2_E = sp.Matrix([sp.Rational(1, 2), 1, 1])

# Transform into the spatial frame s
p1_s = simplify_matrix(G_E_s_config * p1_E)
p2_s = simplify_matrix(G_E_s_config * p2_E)

# Print only the x and y coordinates
for name, point in [("p1", p1_s), ("p2", p2_s)]:
    print(f"\n{name} in frame s (x, y):")
    sp.pprint(point[:2, :].evalf(6))

G_E^s at the given configuration:
⎡-0.965926  -0.258819  1.04118 ⎤
⎢                              ⎥
⎢0.258819   -0.965926  -1.53195⎥
⎢                              ⎥
⎣    0          0        1.0   ⎦

p1 in frame s (x, y):
⎡1.04118 ⎤
⎢        ⎥
⎣-1.53195⎦

p2 in frame s (x, y):
⎡0.299399⎤
⎢        ⎥
⎣-2.36847⎦


In [ ]:
# Part (c): Express the point q^B in the end-effector frame
q_B = sp.Matrix([1, 2, 1])

# Substitute the configuration from part (b), then invert
G_B_1 = simplify_matrix(G_1_B.subs(configuration).inv())
G_1_2 = simplify_matrix(G_2_1.subs(configuration).inv())
G_2_E = simplify_matrix(G_E_2.subs(configuration).inv())

# Convert the same point through the successive frames
q_1 = simplify_matrix(G_B_1 * q_B)
q_2 = simplify_matrix(G_1_2 * q_1)
q_E = simplify_matrix(G_2_E * q_2)

for label, point in [
    ("q in frame 1", q_1),
    ("q in frame 2", q_2),
    ("q in frame E (final)", q_E),
]:
    print(f"\n{label}:")
    sp.pprint(point[:2, :].evalf(6))